# Reproducing the M1 benchmark (Google Colab, CPU-only)

Owner: Hiếu (Member 3) · drafted by Thiên, 27/10/2026. Drafted with the help of an AI assistant (Claude, Anthropic), cited per the course rules.

The official figures come from the full crawl (`colab_crawl.ipynb`, every chunk profiled). This notebook lets anyone **re-run the same measurement from a fresh runtime**, on a whole range (`LIMIT = None`) or on a sample. Use *Runtime → Change runtime type → CPU*, then **Run all**.

- **Time:** end-to-end wall time, discovery time, mean time per paper (`/usr/bin/time -v`, `run_summary.json`).
- **Memory:** peak and mean RAM as the total resident memory of all processes (`src/utils/profiler.py`, sampled every second), peak disk, output size.

Installation time is **not** part of the crawler's wall time.

In [ ]:
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"
BRANCH = "main"
RANGE = "thien"  # range to benchmark
OFFSET = 0
LIMIT = 500  # 500 IDs for benchmark sample
WITH_REFERENCES = True  # measure Semantic Scholar running in parallel
OUT = "/content/bench"

In [ ]:
# Machine configuration, reported in the book
!nproc
!lscpu | grep -E "Model name|^CPU\(s\)"
!free -h
!df -h /content
!python --version
!nvidia-smi -L || echo "No GPU: CPU-only runtime as required"

In [ ]:
import os
import subprocess

if os.path.isdir("/content/CodeRepo/.git"):
    subprocess.run(["git", "-C", "/content/CodeRepo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/CodeRepo"], check=True)
%cd /content/CodeRepo
# Install dependencies without breaking Google Colab's pre-installed packages
!pip install -q --no-deps -r src/requirements.txt || pip install -q -r src/requirements.txt
!test -x /usr/bin/time || (apt-get -qq update && apt-get -qq install -y time)
!git log --oneline -1

In [ ]:
import os

from google.colab import userdata

# Direct configuration fallback if not configured in Colab Secrets:
DIRECT_SECRETS = {
    "S2_API_KEY": "",
    "ARXIV_CONTACT_EMAIL": "",  # Replace with your contact email here if not using Colab Secrets
}

for name in ("S2_API_KEY", "ARXIV_CONTACT_EMAIL"):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = str(val).strip()
            print(f"{name}: set from Colab Secrets")
            continue
    except Exception:
        pass
    if DIRECT_SECRETS.get(name):
        os.environ[name] = DIRECT_SECRETS[name].strip()
        print(f"{name}: set from DIRECT_SECRETS fallback")
    else:
        print(f"{name}: NOT set (using default public mode)")

In [ ]:
SELECT = f"--range {RANGE} --offset {OFFSET}" + (f" --limit {LIMIT}" if LIMIT else "")
REFS = "" if WITH_REFERENCES else "--no-references"
PROFILER = (
    f"python -m src.utils.profiler --out {OUT}/work/profile/bench"
    f" --watch {OUT}/data --watch {OUT}/work --output {OUT}/data/corpus"
)
PIPELINE = (
    f"python -m src.run_pipeline {SELECT} {REFS}"
    f" --data-root {OUT}/data/corpus --work-dir {OUT}/work --no-progress"
)
!rm -rf {OUT} && mkdir -p {OUT}/data {OUT}/work/profile/bench
!/usr/bin/time -v -o {OUT}/work/profile/bench/time_v.log {PROFILER} -- {PIPELINE}

In [ ]:
import json
import re
from pathlib import Path

prof = json.loads(Path(OUT, "work", "profile", "bench", "profile_summary.json").read_text())
run = json.loads(Path(OUT, "work", "run_summary.json").read_text())
time_v = Path(OUT, "work", "profile", "bench", "time_v.log").read_text()


def from_time_v(label):
    m = re.search(rf"{re.escape(label)}: (.+)", time_v)
    return m.group(1).strip() if m else None


report = {
    "wall time end-to-end (profiler, s)": prof["wall_s"],
    "wall time (/usr/bin/time)": from_time_v("Elapsed (wall clock) time (h:mm:ss or m:ss)"),
    "discovery (s)": run["discovery_s"],
    "mean time per paper (s)": run["avg_paper_s"],
    "peak RAM, all processes (MB)": prof["rss_max_mb"],
    "mean RAM (MB)": prof["rss_mean_mb"],
    "peak RAM of the main process (/usr/bin/time, kB)": from_time_v(
        "Maximum resident set size (kbytes)"
    ),
    "peak disk (MB)": prof["disk_max_mb"],
    "output (MB)": prof["output_mb"],
    "papers complete": f"{run['papers_complete']}/{run['ids_requested']}",
    "smallest gap between two arXiv requests (s)": run["min_request_gap_s"],
    "machine": f"{prof['cpu_count']} vCPU, {prof['ram_total_mb']} MB RAM",
}
for k, v in report.items():
    print(f"{k:<50} {v}")

In [ ]:
# CSV files for the book (src/analysis/stats.py) and an archive to download
from google.colab import files

STATS = (
    f"python -m src.analysis.stats --data-root {OUT}/data/corpus --work-dir {OUT}/work"
    f" --out {OUT}/book_csv"
)
KEEP = "book_csv work/profile work/run_summary.json work/runs.jsonl work/run.log"
!{STATS}
!cd {OUT} && zip -qr bench_results.zip {KEEP}
files.download(f"{OUT}/bench_results.zip")